## Product Sales 
Focus MAterial based insights but additionsla filters available


In [0]:
%sql
WITH best_product_sales AS (
    SELECT
        product_id,
        Round(SUM(total_net_revenue)) AS total_net_revenue,
        Round(SUM(total_pieces_sold)) AS total_pieces,
        MAX(terms) AS terms,
        MAX(material) AS material,
        MAX(origin) AS origin,
        MAX(section) AS section,
        MAX(season) AS season
    FROM zara_lakehouse.retail_gold.gold_product_attribute_sales
    GROUP BY product_id
    ORDER BY total_net_revenue DESC
    ) select * from best_product_sales

#### Flexible Material - Monthly Sales pattern

In [0]:
%sql
SELECT
    -- CHANGE DIMENSION HERE
    material,
    MONTH(date) AS month,

    SUM(total_net_revenue) AS total_revenue,
    SUM(total_pieces_sold) AS total_volume,
    AVG(avg_discount) AS avg_discount

FROM zara_lakehouse.retail_gold.gold_product_attribute_sales

WHERE 1=1
    -- MODIFY FILTERS
    -- AND section = 'WOMAN'
    -- AND season = 'Winter'
    -- AND material = 'Polyester'

GROUP BY 
    material,
    MONTH(date)

ORDER BY 
    material,
    month;

Databricks visualization. Run in Databricks to view.

#### Flexible Material  Sales pattern

In [0]:
%sql
WITH product_sales AS (
    SELECT
        product_id,
        SUM(total_net_revenue) AS total_revenue,
        SUM(total_pieces_sold) AS total_pieces,
        MAX(material) AS material,
        MAX(terms) AS terms
    FROM zara_lakehouse.retail_gold.gold_product_attribute_sales
    WHERE 1=1
        -- OPTIONAL FILTERS
        -- AND section = 'WOMAN'
        -- AND season = 'Winter'
    GROUP BY product_id
)

SELECT
    material,

    COUNT(*) AS product_count,
    AVG(total_revenue) AS avg_revenue,
    AVG(total_pieces) AS avg_volume,

    --  key metric
    AVG(total_revenue / NULLIF(total_pieces,0)) AS revenue_per_unit,

    -- ranking
    RANK() OVER (ORDER BY AVG(total_revenue) DESC) AS material_rank

FROM product_sales
GROUP BY material
ORDER BY material_rank;

#### Flexible Material Best and worst Ranks product

In [0]:
%sql
WITH product_sales AS (
    SELECT
        product_id,
        SUM(total_net_revenue) AS total_revenue,
        MAX(material) AS material,
        MAX(terms) AS terms,
        MAX(season) AS season
    FROM zara_lakehouse.retail_gold.gold_product_attribute_sales
    WHERE 1=1
        -- OPTIONAL FILTERS
        -- AND material = 'Wool'
        -- AND season = 'Spring'
    GROUP BY product_id
),

ranked AS (
    SELECT *,
        RANK() OVER (ORDER BY total_revenue DESC) AS best_rank,
        RANK() OVER (ORDER BY total_revenue ASC) AS worst_rank
    FROM product_sales
)

SELECT *
FROM ranked
WHERE best_rank <= 3 OR worst_rank <= 3;